# 09 · Dritte Domäne: Windkraft

Gleiches Experiment wie 07 (Strom), aber auf stündlicher **Windleistung Onshore** (MW).
Hypothese: Die FM-Fehlerkorrelation liegt zwischen Strom (ρ≈0.60) und Volatilität (ρ≈0.97),
weil Wind wetterbedingt chaotischer als Strom, aber strukturierter als Crypto-Volatilität ist.

- **Daten:** Windleistung Onshore (MW), DE/FR/ES/PL (windbetonte Länder), Energy-Charts-API (Fraunhofer ISE, CC BY 4.0)
- **Experten:** dieselben 4 FMs + SNaive + SeasReg wie in 07
- **Gate:** nur Zeitreihe (24 h) + Disagree, kein Feature Engineering
- **Metrik:** Skalierter MAE = |y−ŷ| / Mittelniveau — robuster als MAPE bei kleinen Windwerten

**Kaggle:** Accelerator = GPU T4, Internet = On, dann Run All.

In [ ]:
import os, subprocess, sys
from pathlib import Path

if Path("/kaggle/working").exists():
    repo = Path("/kaggle/working/repo")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/ihringerlinus/Learning_to_Forecast_Time-Series_with_Multiple_Experts.git",
                        str(repo)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "chronos-forecasting", "timesfm[torch]",
                    "tirex-ts", "granite-tsfm", "pyarrow"], check=True)
    os.chdir(repo / "src")
elif not Path("pipeline").exists() and Path("src/pipeline").exists():
    os.chdir("src")
print("Arbeitsordner:", Path.cwd())

In [ ]:
import json, time
from functools import partial
from statistics import NormalDist

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import torch
from tqdm.auto import tqdm

from pipeline import common as C, experts as ex, features as F, combine as cb

DATASET = "wind_1h"
COUNTRIES = ["de", "fr", "es", "pl"]   # windbetonte Länder; PL statt IT (IT hat kaum Onshore-Wind)
FIRST_DAY, LAST_DAY = "2024-07-01", "2026-02-28"  # Test ab März 2026 bleibt zu
CHUNK_MONTHS = 6

C.ORIGIN_START = "2025-01-01"
EVAL_START, EVAL_END, TUNE_DAYS = "2025-11-01", "2026-02-28", 90
if torch.cuda.is_available():
    C.FM_BATCH = 512

FMS = ["Chronos-2-full", "TimesFM-2.5", "TiRex", "FlowState"]
CLASSIC = ["SNaive", "SeasReg"]
MODELS = ["RW"] + CLASSIC + FMS
POOLS = {
    "2 FMs (alt)": ["Chronos-2-full", "TimesFM-2.5"],
    "4 FMs": FMS,
    "4 FMs + klassisch": FMS + CLASSIC,
}
RAW_LAGS = 24
LAMBDAS = [1e-4, 1e-3, 1e-2, 1e-1, 1.0]
ONLY_CACHE = not Path("/kaggle/working").exists()

DATA_DIR = (Path("data") / DATASET).resolve()
RAW_DIR  = (Path("data") / "raw" / "energy_charts").resolve()   # Cache mit Strom teilen (gleiche API)
FC_DIR   = (Path("results") / "forecasts" / DATASET).resolve()
for d in (DATA_DIR, RAW_DIR, FC_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("Gerät:", ex._device(), "| nur Cache:", ONLY_CACHE)
print("Cache:", len(list(DATA_DIR.glob("*.parquet"))), "Länder,",
      len(list(FC_DIR.glob("*__*.parquet"))), "Prognosedateien,",
      "Gate-Ergebnisse" if (FC_DIR.parent / "09_res.parquet").exists() else "keine Gate-Ergebnisse")

## 1 · Daten

In [ ]:
API = "https://api.energy-charts.info/public_power"


def fetch(country, start, end):
    path = RAW_DIR / f"{country}_{start}_{end}.json"   # Cache wird mit 07 geteilt
    if path.exists():
        return json.loads(path.read_text())
    while True:
        r = requests.get(API, params={"country": country, "start": start, "end": end}, timeout=180)
        if r.status_code == 429:
            wait = int(r.headers.get("Retry-After", 60))
            print(f"  Rate-Limit, warte {wait}s")
            time.sleep(wait)
            continue
        r.raise_for_status()
        break
    path.write_text(r.text)
    time.sleep(31)
    return r.json()


def hourly_wind(country):
    """Windleistung Onshore (MW), stündlich. Spalte rv = Windleistung (wie rv=Volatilität in anderen Notebooks)."""
    path = DATA_DIR / f"{country}_daily.parquet"
    if path.exists():
        return pd.read_parquet(path)
    parts = []
    for a in pd.date_range(FIRST_DAY, LAST_DAY, freq=f"{CHUNK_MONTHS}MS"):
        b = min(a + pd.DateOffset(months=CHUNK_MONTHS) - pd.Timedelta(days=1), pd.Timestamp(LAST_DAY))
        js = fetch(country, a.strftime("%Y-%m-%d"), b.strftime("%Y-%m-%d"))
        wind = next((p["data"] for p in js["production_types"] if p["name"] == "Wind onshore"), None)
        if wind is None:
            avail = [p["name"] for p in js["production_types"]]
            raise ValueError(f"{country}: keine Reihe 'Wind onshore', vorhanden: {avail}")
        parts.append(pd.Series(np.array(wind, dtype=float),
                               index=pd.to_datetime(js["unix_seconds"], unit="s")))
        print(f"{country}: {a.date()} bis {b.date()} geladen")
    s = pd.concat(parts).sort_index()
    h = s[~s.index.duplicated()].resample("1h").mean().dropna()
    # Stunden mit Windleistung <= 0 entfernen (Datenfehler / Netzeinspeisung negativ)
    h = h[h > 0]
    n_removed = len(s.resample("1h").mean()) - len(h)
    if n_removed > 0:
        print(f"{country}: {n_removed} Stunden mit Wind <= 0 entfernt ({100*n_removed/len(h):.1f}%)")
    d = pd.DataFrame({"rv": h, "ret": np.log(h).diff().fillna(0.0)}).rename_axis("date")
    d.to_parquet(path)
    return d


daily = {c: hourly_wind(c) for c in COUNTRIES}
(Path("data") / f"{DATASET}.json").write_text(json.dumps(
    {"name": DATASET, "sids": COUNTRIES, "daily_dir": str(DATA_DIR), "forecast_dir": str(FC_DIR),
     "test_start": "2026-03-01",
     "info": {"source": "energy-charts.info public_power, Wind onshore",
              "note": "Metrik: Skalierter MAE = |y-f| / E[y], nicht MAPE (robust gegen kleine Windwerte)"}}, indent=2))

fig, ax = plt.subplots(figsize=(11, 3.5))
for c, d in daily.items():
    ax.plot(d["rv"].resample("D").mean() / 1000, lw=0.7, label=f"{c} ({len(d):,} h)")
ax.axvspan(pd.Timestamp(EVAL_START), pd.Timestamp(EVAL_END), color="grey", alpha=0.15)
ax.set_title("Windleistung Onshore in GW (Tagesmittel), grau = Evaluation", fontsize=10)
ax.legend(fontsize=8)
plt.show()

## 2 · Experten

Identisch zu Notebook 07. SNaive und SeasReg sind generische Zeitreihenmodelle und funktionieren auch für Windleistung.

In [ ]:
def from_log9(logq9):
    logq9 = np.sort(np.asarray(logq9, float)[:, [h - 1 for h in C.HORIZONS], :], axis=-1)
    q = np.sort(np.exp(ex.expand_quantiles(logq9)), axis=-1)
    return C.mean_from_quantiles(q), q


class SeasonalNaive(ex.Expert):
    name = "SNaive"

    def forecast(self, s, origins):
        L = s.logrv
        mean, q = self._alloc(len(origins))
        for i, t in enumerate(origins):
            for j, h in enumerate(C.HORIZONS):
                u = np.arange(max(168, t - C.EST_WINDOW + 1), t - h + 1)
                resid = L[u + h] - L[u + h - 168]
                mean[i, j], q[i, j] = C.dist_from_log_resid(L[t + h - 168], resid)
        return mean, q


class SeasonalRegression(ex.Expert):
    name = "SeasReg"

    def forecast(self, s, origins):
        L = s.logrv
        m24 = pd.Series(L).rolling(24).mean().to_numpy()
        mean, q = self._alloc(len(origins))
        for j, h in enumerate(C.HORIZONS):
            X = lambda u: np.column_stack([np.ones(len(u)), L[u], L[u + h - 24], L[u + h - 168], m24[u]])
            for i, t in enumerate(origins):
                u = np.arange(max(168, t - C.EST_WINDOW + 1), t - h + 1)
                Xu = X(u)
                beta, *_ = np.linalg.lstsq(Xu, L[u + h], rcond=None)
                mean[i, j], q[i, j] = C.dist_from_log_resid((X(np.array([t])) @ beta)[0], L[u + h] - Xu @ beta)
        return mean, q


class TiRex(ex.Expert):
    name = "TiRex"

    def __init__(self, chunk=1024):
        from tirex import load_model
        self.chunk = chunk
        self.model = load_model("NX-AI/TiRex", backend="torch", device=ex._device())

    def forecast(self, s, origins):
        ctx = np.stack(ex.contexts_for(s, origins))
        parts = []
        for k in tqdm(range(0, len(ctx), self.chunk), desc=f"TiRex {s.sid}", leave=False):
            q, _ = self.model.forecast(context=ctx[k:k + self.chunk], prediction_length=max(C.HORIZONS),
                                       output_type="numpy", batch_size=256)
            parts.append(q)
        return from_log9(np.concatenate(parts))


class FlowState(ex.Expert):
    name = "FlowState"

    def __init__(self, batch=64, scale_factor=1.0):
        from tsfm_public import FlowStateForPrediction
        self.batch, self.scale = batch, scale_factor
        self.model = (FlowStateForPrediction.from_pretrained("ibm-granite/granite-timeseries-flowstate-r1")
                      .to(ex._device()).eval())

    def forecast(self, s, origins):
        ctx = np.stack(ex.contexts_for(s, origins))
        parts = []
        for k in tqdm(range(0, len(ctx), self.batch), desc=f"FlowState {s.sid}", leave=False):
            x = torch.from_numpy(ctx[k:k + self.batch])[:, :, None].to(ex._device())
            with torch.no_grad():
                out = self.model(x, scale_factor=self.scale, prediction_length=max(C.HORIZONS), batch_first=True)
            parts.append(out.quantile_outputs[..., 0].cpu().numpy().transpose(0, 2, 1))
        return from_log9(np.concatenate(parts))


EXPERTS = {**ex.EXPERTS, "SNaive": SeasonalNaive, "SeasReg": SeasonalRegression, "TiRex": TiRex, "FlowState": FlowState}
series = {c: ex.RVSeries(c, daily[c]) for c in COUNTRIES}
print({c: len(x.origins) for c, x in series.items()}, "Ursprünge")

for m in MODELS:
    todo = [c for c in COUNTRIES if not (FC_DIR / f"{c}__{m}.parquet").exists()]
    if not todo:
        print(f"{m:>15} | gecacht")
        continue
    if ONLY_CACHE:
        print(f"{m:>15} | fehlt im Cache für {todo}, übersprungen (ONLY_CACHE = False zum Rechnen)")
        continue
    try:
        expert = EXPERTS[m]()
    except ImportError as e:
        print(f"{m:>15} | übersprungen, Paket fehlt ({e.name})")
        continue
    for c in todo:
        t0, s = time.time(), series[c]
        mean, q = expert.forecast(s, s.origins)
        ex.forecast_frame(m, s, s.origins, mean, q).to_parquet(FC_DIR / f"{c}__{m}.parquet")
        print(f"{m:>15} | {c}: {time.time() - t0:.0f}s")
    del expert
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 3 · Wie gut sind die einzelnen Experten? (Evaluation Nov 25 – Feb 26)

Metrik: Skalierter MAE = |y−ŷ| / E[y] in %. Robuster als MAPE bei kleinen Windwerten.

In [ ]:
fc_all = C.cap_quantiles(pd.concat([pd.read_parquet(p) for p in sorted(FC_DIR.glob("*__*.parquet"))], ignore_index=True))
MODELS = [m for m in MODELS if m in set(fc_all["model"])]
POOLS = {k: [m for m in v if m in MODELS] for k, v in POOLS.items()}
print("Pools:", POOLS)

ev = fc_all[(fc_all["target_date"] >= EVAL_START) & (fc_all["target_date"] < pd.Timestamp(EVAL_END) + pd.Timedelta(days=1))].copy()

# Skalierter MAE: Fehler relativ zum mittleren Niveau je Serie und Horizont
mean_level = ev.groupby(["h", "sid"])["y"].transform("mean")
ev["smae"] = 100 * np.abs(ev["y"] - ev["mean"]) / mean_level
ev["logerr"] = np.log(ev["y"] / ev["mean"])

print("Skalierter MAE in % (kleiner = besser)")
display(ev.pivot_table(index=["h", "sid"], columns="model", values="smae")[MODELS].round(2))

for h in C.HORIZONS:
    E = ev[ev["h"] == h].pivot_table(index=["sid", "origin"], columns="model", values="logerr")
    print(f"h={h}: Korrelation der Log-Fehler")
    display(E[[m for m in MODELS if m != "RW"]].corr().round(2))

## 4 · Gate nur auf Zeitreihe + Disagree

In [ ]:
def wind_loss(w, mean, Q, y, s, alpha=None):
    """Wie load_loss in 07: quadratischer Fehler relativ zum Niveau + Pinball."""
    alpha = cb.ALPHA if alpha is None else alpha
    f = (w * mean).sum(1)
    d = y[:, None] - torch.einsum("nm,nmq->nq", w, Q)
    lv = torch.as_tensor(C.LV, dtype=cb.DTYPE)
    pin = torch.maximum(lv * d, (lv - 1) * d).mean(1)
    return (((y - f) / s) ** 2).mean() + alpha * (pin / s).mean()


cb.comb_loss_torch = wind_loss
# Auswahl nach skaliertem Fehler (robust gegen windstille Stunden)
cb.qlike = lambda y, f: np.abs(y - f) / np.maximum(np.mean(y), 1e-6)


def raw_features(d):
    lrv = np.log(d["rv"])
    norm = np.log(d["rv"].rolling(7 * 24).mean())
    f = pd.DataFrame({f"raw_{i}": lrv.shift(i) - norm for i in range(RAW_LAGS)})
    f["scale"] = d["rv"].rolling(22 * 24).mean()
    return f


daily_feat = pd.concat({c: raw_features(daily[c]) for c in COUNTRIES}, names=["sid", "origin"])


def make_panels(pool):
    cfg = F.FeatureConfig(pool, COUNTRIES, groups=["market", "disagree"], calendar=[])
    fc = F.add_recent(fc_all[fc_all["model"].isin(pool)], cfg)
    panels = {}
    for h in C.HORIZONS:
        df, groups = F.build_panel(fc, daily_feat, h, cfg)
        groups = {("raw" if g == "market" else g): v for g, v in groups.items()}
        panels[h] = cb.Panel(df, h, pool, groups)
    return panels


METHODS = [
    cb.Method("Gleichgewicht", cb.Equal),
    cb.Method("Auswahl", cb.BestSingle),
    cb.Method("Statisch", partial(cb.Gate, groups=())),
    cb.Method("Gate linear", cb.Gate, {"lam": LAMBDAS}),
    cb.Method("Gate linear [nur disagree]", partial(cb.Gate, groups=("disagree",)), {"lam": LAMBDAS}),
    cb.Method("Gate MLP", partial(cb.Gate, hidden=16), {"lam": LAMBDAS}),
]

RES_PATH, WTS_PATH = FC_DIR.parent / "09_res.parquet", FC_DIR.parent / "09_wts.parquet"
if RES_PATH.exists():
    res, wts = pd.read_parquet(RES_PATH), pd.read_parquet(WTS_PATH)
    print("Gate-Ergebnisse aus Cache geladen (zum Neurechnen 09_res/09_wts löschen)")
else:
    rows, wts = [], []
    t0 = time.time()
    for pool_name, pool in POOLS.items():
        for h, P in make_panels(pool).items():
            for fold in cb.walkforward_folds(P, EVAL_START, EVAL_END, "MS", TUNE_DAYS):
                Pt = P.take(fold.test)
                base = Pt.meta[["sid", "h", "target_date", "y"]].assign(pool=pool_name, fold=fold.label)
                for j, m in enumerate(pool):
                    rows.append(base.assign(model=m, mean=Pt.mean[:, j]))
                for method in METHODS:
                    c, best, _ = cb.fit_predict(P, method, fold)
                    w = c.weights(Pt)
                    rows.append(base.assign(model=method.name, mean=(w * Pt.mean).sum(1)))
                    wts.append(pd.DataFrame(w, columns=pool).assign(pool=pool_name, h=h, model=method.name))
            print(f"{pool_name:<20} h={h:<3} fertig ({time.time() - t0:.0f}s)")
    res = pd.concat(rows, ignore_index=True)
    wts = pd.concat(wts, ignore_index=True)
    res.to_parquet(RES_PATH)
    wts.to_parquet(WTS_PATH)

# Skalierter MAE
res_mean_level = res.groupby(["pool", "h", "sid"])["y"].transform("mean")
res["smae"] = 100 * np.abs(res["y"] - res["mean"]) / res_mean_level

## 5 · Ergebnisse

Skalierter MAE relativ zum besten Einzelexperten des Pools (ex post), negativ = besser.
p: Diebold-Mariano auf dem stündlichen Mittel über die Länder (Newey-West, Lag max(h−1, 24)).

In [ ]:
def dm_p(d, h):
    d = d[np.isfinite(d)]
    T, u, L = len(d), d - d.mean(), max(h - 1, 24)
    var = u @ u / T + 2 * sum((1 - l / (L + 1)) * (u[l:] @ u[:-l]) / T for l in range(1, L + 1))
    with np.errstate(invalid="ignore", divide="ignore"):
        return 2 * (1 - NormalDist().cdf(abs(d.mean() / np.sqrt(var / T))))


out = []
for (pool_name, h), g in res.groupby(["pool", "h"], sort=False):
    L = g.pivot_table(index="target_date", columns="model", values="smae")
    best = L[POOLS[pool_name]].mean().idxmin()
    for m in L.columns:
        out.append({"Pool": pool_name, "Modell": m, "h": h, "sMAE": L[m].mean(),
                    "Δ % vs. bester": 100 * (L[m].mean() / L[best].mean() - 1),
                    "p": np.nan if m == best else dm_p((L[m] - L[best]).to_numpy(), h)})
    print(f"{pool_name:<20} h={h:<3} bester Experte: {best} (sMAE {L[best].mean():.2f} %)")

S = pd.DataFrame(out)
order = list(dict.fromkeys(MODELS + [m.name for m in METHODS]))
S["Modell"] = pd.Categorical(S["Modell"], order)
S["Pool"] = pd.Categorical(S["Pool"], list(POOLS))
display(S.pivot_table(index=["Pool", "Modell"], columns="h", values=["Δ % vs. bester", "p"], observed=True).round(3))

In [ ]:
# Pro Land: Δ % sMAE vs. bester Experte
out = []
for (pool_name, h, sid), g in res.groupby(["pool", "h", "sid"], sort=False):
    Lm = g.groupby("model")["smae"].mean()
    best = Lm[POOLS[pool_name]].min()
    for m in ["Gleichgewicht", "Statisch", "Gate linear", "Gate MLP"]:
        out.append({"Pool": pool_name, "Land": sid, "h": h, "Modell": m, "Δ %": 100 * (Lm[m] / best - 1)})
display(pd.DataFrame(out).pivot_table(index=["Pool", "Land"], columns=["h", "Modell"], values="Δ %").round(2))

for pool_name, pool in POOLS.items():
    w = wts[(wts["pool"] == pool_name) & wts["model"].isin(["Statisch", "Gate linear", "Gate MLP"])]
    print(pool_name)
    display(w.groupby(["h", "model"])[pool].mean().round(3))

## 6 · Obergrenze: Was wäre mit Kombinieren im besten Fall möglich?

Feste Gewichte und Gate, direkt auf der Evaluationsperiode optimiert.

In [ ]:
for pool_name, pool in POOLS.items():
    for h, P in make_panels(pool).items():
        Pe = P.take(np.asarray((P.td >= EVAL_START) & (P.td < pd.Timestamp(EVAL_END) + pd.Timedelta(days=1))))
        level = Pe.y.mean()
        A = np.abs(Pe.y[:, None] - Pe.mean) / level
        best = A.mean(0).min()
        smae = lambda c: (np.abs(Pe.y - (c.weights(Pe) * Pe.mean).sum(1)) / level).mean()
        print(f"{pool_name:<18} h={h:<3}"
              f" feste Gewichte ex post: {100 * (smae(cb.Gate(groups=()).fit(Pe)) / best - 1):+.2f} %"
              f" | Gate ex post: {100 * (smae(cb.Gate(lam=1e-3).fit(Pe)) / best - 1):+.2f} %"
              f" | Oracle: {100 * (A.min(1).mean() / best - 1):+.1f} %")

In [ ]:
# Nur Kaggle: Ergebnisse als ZIP exportieren
if Path("/kaggle/working").exists():
    import shutil
    out = Path("/kaggle/working/09_out")
    shutil.copytree(FC_DIR, out / "results/forecasts" / DATASET, dirs_exist_ok=True)
    shutil.copytree(DATA_DIR, out / "data" / DATASET, dirs_exist_ok=True)
    shutil.copy(Path("data") / f"{DATASET}.json", out / "data")
    for p in (RES_PATH, WTS_PATH):
        shutil.copy(p, out / "results/forecasts")
    shutil.make_archive("/kaggle/working/09_ergebnisse", "zip", out)
    print("fertig: /kaggle/working/09_ergebnisse.zip")